# 13 · GNN

Template para `RentalPriceGNN`, con foco en construcción del grafo, evaluación comparable y análisis de qué aprende la red.

## Hipótesis del modelo

- El precio surge de atributos del inmueble y de la estructura relacional entre observaciones cercanas.
- La definición del grafo es parte del modelo y debe documentarse con tanto cuidado como los hiperparámetros.
- La interpretación se apoya más en ablations, embeddings y atención que en importancias clásicas.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import geopandas as gpd
import seaborn as sns


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from ml_core.models.gat_gcn_model import GraphAttentionGCN
from ml_core.preprocessing.knhs import KNHS
from sklearn.neighbors import BallTree
from ml_core.evaluation.modelEvaluator import regression_metrics
from ml_core.visualization.mapper import *



OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "13_gnn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


## Datos base

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "splits"

train_raw = pd.read_csv(DATA_PATH / "arg_venta_data_train.csv")
gdf_train = gpd.GeoDataFrame(
    train_raw,
    geometry=gpd.points_from_xy(
        train_raw["longitud"],
        train_raw["latitud"]
    ),
    crs="EPSG:4326"
)

test_raw = pd.read_csv(DATA_PATH / "arg_venta_data_test.csv")
gdf_test = gpd.GeoDataFrame(
    test_raw,
    geometry=gpd.points_from_xy(
        test_raw["longitud"],
        test_raw["latitud"]
    ),
    crs="EPSG:4326"
)

val_raw = pd.read_csv(DATA_PATH / "arg_venta_data_val.csv")
gdf_val = gpd.GeoDataFrame(
    val_raw,
    geometry=gpd.points_from_xy(
        val_raw["longitud"],
        val_raw["latitud"]
    ),
    crs="EPSG:4326"
)

target_col = "log_precio"   
coord_cols = ["longitud", "latitud"]
feature_cols = [
    'area_m2_total_scaled',
    'area_m2_descubierta_scaled',
    'ambientes_scaled',
    'antiguedad_scaled',
    'expensas_scaled',
    'banos_scaled',
    'cocheras_scaled',
    'estado_num',
    'disposicion_Frente',
    'disposicion_Contrafrente',
    'disposicion_Lateral',
    'dist_subte_scaled',
    'dist_universidad_scaled',
    'dist_hospital_scaled',
    'dist_est_educativo_scaled',
    'dist_espacio_verde_scaled',
    'dist_areas_programaticas_scaled',
    'dist_avenida_rivadavia_scaled',
    "n_robos_1000m_scaled",
    "n_universidades_1000m_scaled",
    'pozo'
]


## Definición del grafo

Dejá explícito:

- cómo conectás nodos
- si usás KNN, radio, barrio o una mezcla
- qué atributos van en `edge_attr`
- si el grafo se arma solo con train o con todo el dataset

In [3]:
X_train = gdf_train[feature_cols]
y_train = gdf_train[target_col]
coords_train = gdf_train[coord_cols].to_numpy()

X_test = gdf_test[feature_cols]
y_test = gdf_test[target_col]
coords_test = gdf_test[coord_cols].to_numpy()

X_val = gdf_val[feature_cols]
y_val = gdf_val[target_col]
coords_val = gdf_val[coord_cols].to_numpy()

In [4]:
# Hyperparámetros de grafo
k_neighbors = 15  # vecinos por nodo para construir aristas
radius_km = 3.0
bandwidth_km = 2.0  # para pesos locales (LGWR aproximada)
weight_cols = [f"w_{c}" for c in feature_cols]


def compute_local_feature_weights(X_df, y_array, coords_rad, bandwidth_km=2.0, k_neighbors=50):
    """Estima pesos por feature usando regresión local ponderada espacialmente."""
    from sklearn.neighbors import BallTree

    X = np.asarray(X_df.to_numpy(), dtype=float)
    y = np.asarray(y_array, dtype=float).reshape(-1)
    n, d = X.shape
    tree = BallTree(coords_rad, metric="haversine")
    weights = np.zeros((n, d), dtype=float)

    for i in range(n):
        dist, idx = tree.query(coords_rad[i:i+1], k=min(k_neighbors + 1, n))
        dist = dist.ravel() * 6371.0
        idx = idx.ravel()
        mask = dist > 0
        dist, idx = dist[mask], idx[mask]
        if len(idx) == 0:
            weights[i] = 1.0
            continue

        kernel = np.exp(-(dist ** 2) / (bandwidth_km ** 2))
        Xn = X[idx]
        A = Xn * kernel[:, None]
        AtY = A.T @ y[idx]

        beta = None
        ridge = 1e-6
        for _ in range(6):
            AtX = A.T @ Xn + np.eye(d) * ridge
            try:
                beta = np.linalg.solve(AtX, AtY)
                break
            except np.linalg.LinAlgError:
                ridge *= 10.0

        if beta is None:
            AtX = A.T @ Xn + np.eye(d) * ridge
            beta = np.linalg.pinv(AtX) @ AtY

        beta = np.nan_to_num(beta, nan=0.0, posinf=0.0, neginf=0.0)
        weights[i] = np.maximum(np.abs(beta), 1e-8)

    return weights


def project_local_feature_weights(weights_source, coords_source, coords_target, bandwidth_km, k_neighbors):
    tree = BallTree(np.asarray(coords_source), metric="haversine")
    k_proj = min(k_neighbors, len(coords_source))
    dist_proj, idx_proj = tree.query(np.asarray(coords_target), k=k_proj)
    kernel_proj = np.exp(-((dist_proj * 6371.0) ** 2) / (bandwidth_km ** 2))
    kernel_proj = kernel_proj / (kernel_proj.sum(axis=1, keepdims=True) + 1e-9)
    return (kernel_proj[..., None] * weights_source[idx_proj]).sum(axis=1)


def build_weighted_graph_frame(X_df, coords_rad, weights):
    graph = X_df.copy().reset_index(drop=True)
    graph["lat_deg"] = np.rad2deg(coords_rad[:, 0])
    graph["lon_deg"] = np.rad2deg(coords_rad[:, 1])
    for col, vals in zip(weight_cols, weights.T):
        graph[col] = vals
    return graph


def build_cross_split_knhs_graph(builder, graph_source, graph_target):
    """Arma un grafo combinado train+target con aristas train->target.

    Los nodos target eligen vecinos en train con la misma lógica KNHS.
    Agregamos solo aristas train->target para evitar que target altere las
    embeddings de train durante la inferencia.
    """
    edge_index_source, edge_attr_source = builder.build(graph_source)

    coords_source = graph_source[[builder.lat_col, builder.lon_col]].to_numpy(dtype=float)
    coords_target = graph_target[[builder.lat_col, builder.lon_col]].to_numpy(dtype=float)
    feats_source = graph_source[builder.feature_cols].to_numpy(dtype=float)
    feats_target = graph_target[builder.feature_cols].to_numpy(dtype=float)
    weights_source = graph_source[builder.weight_cols].to_numpy(dtype=float)
    weights_target = graph_target[builder.weight_cols].to_numpy(dtype=float)

    lat_source = np.deg2rad(coords_source[:, 0])
    lon_source = np.deg2rad(coords_source[:, 1])
    lat_target = np.deg2rad(coords_target[:, 0])
    lon_target = np.deg2rad(coords_target[:, 1])

    src_list = [int(x) for x in edge_index_source[0]]
    dst_list = [int(x) for x in edge_index_source[1]]
    attr_list = edge_attr_source.tolist()

    target_offset = len(graph_source)
    for i in range(len(graph_target)):
        d_km = builder._haversine_km(lat_target[i], lon_target[i], lat_source, lon_source)
        candidate_idx = np.nonzero(d_km <= builder.radius_km)[0]
        if len(candidate_idx) == 0:
            continue

        if builder.distance == "euclidean":
            feat_d = np.linalg.norm(feats_source[candidate_idx] - feats_target[i], axis=1)
        else:
            w_mean = 0.5 * (weights_source[candidate_idx] + weights_target[i])
            diff = feats_source[candidate_idx] - feats_target[i]
            feat_d = np.sqrt(np.sum(w_mean * diff * diff, axis=1))

        top_k = np.argsort(feat_d)[: builder.k]
        neighbors = candidate_idx[top_k]
        target_idx = target_offset + i
        for j, dist_feat in zip(neighbors, feat_d[top_k]):
            src_list.append(int(j))
            dst_list.append(int(target_idx))
            attr_list.append([float(d_km[j]), float(dist_feat)])

    combined_graph = pd.concat([graph_source, graph_target], ignore_index=True)
    edge_index = np.vstack([src_list, dst_list]).astype(np.int64)
    edge_attr = np.asarray(attr_list, dtype=np.float32)
    target_mask = np.zeros(len(combined_graph), dtype=bool)
    target_mask[target_offset:] = True
    return combined_graph, edge_index, edge_attr, target_mask


weights_train = compute_local_feature_weights(
    X_train,
    y_train,
    coords_train,
    bandwidth_km=bandwidth_km,
    k_neighbors=k_neighbors,
)
weights_val = project_local_feature_weights(
    weights_train,
    coords_train,
    coords_val,
    bandwidth_km=bandwidth_km,
    k_neighbors=k_neighbors,
)
weights_test = project_local_feature_weights(
    weights_train,
    coords_train,
    coords_test,
    bandwidth_km=bandwidth_km,
    k_neighbors=k_neighbors,
)

graph_train = build_weighted_graph_frame(X_train, coords_train, weights_train)
graph_val = build_weighted_graph_frame(X_val, coords_val, weights_val)
graph_test = build_weighted_graph_frame(X_test, coords_test, weights_test)

builder = KNHS(
    lat_col="lat_deg",
    lon_col="lon_deg",
    feature_cols=feature_cols,
    weight_cols=weight_cols,
    distance="local_weighted",
    radius_km=radius_km,
    k=k_neighbors,
    add_reverse=True,
)

# Metodología 1: cada split construye su propio grafo con KNHS.
edge_index_train, edge_attr_train = builder.build(graph_train)
edge_index_val, edge_attr_val = builder.build(graph_val)
edge_index_test, edge_attr_test = builder.build(graph_test)
print("[split-local] Aristas train:", edge_index_train.shape[1])
print("[split-local] Aristas val:", edge_index_val.shape[1])
print("[split-local] Aristas test:", edge_index_test.shape[1])

# Metodología 2: val/test se conectan contra vecinos de train con KNHS cross-split.
graph_val_cross, edge_index_val_cross, edge_attr_val_cross, val_cross_mask = build_cross_split_knhs_graph(
    builder,
    graph_train,
    graph_val,
)
graph_test_cross, edge_index_test_cross, edge_attr_test_cross, test_cross_mask = build_cross_split_knhs_graph(
    builder,
    graph_train,
    graph_test,
)
X_val_cross = pd.concat([X_train.reset_index(drop=True), X_val.reset_index(drop=True)], ignore_index=True)
y_val_cross = pd.concat([y_train.reset_index(drop=True), y_val.reset_index(drop=True)], ignore_index=True)
X_test_cross = pd.concat([X_train.reset_index(drop=True), X_test.reset_index(drop=True)], ignore_index=True)
y_test_cross = pd.concat([y_train.reset_index(drop=True), y_test.reset_index(drop=True)], ignore_index=True)
print("[cross-split] Aristas val:", edge_index_val_cross.shape[1])
print("[cross-split] Aristas test:", edge_index_test_cross.shape[1])
print("[cross-split] Nodos target en val:", int(val_cross_mask.sum()))
print("[cross-split] Nodos target en test:", int(test_cross_mask.sum()))



[split-local] Aristas train: 751314
[split-local] Aristas val: 56430
[split-local] Aristas test: 57502
[cross-split] Aristas val: 831409
[cross-split] Aristas test: 831550
[cross-split] Nodos target en val: 7443
[cross-split] Nodos target en test: 7507


## Fine-tuning

ACordarse, el wrapper de models tiene ya un métodod e fine_tuning, deberíamos hacer que este choclo que definimos acá quede dentro de la clase correspondiente.

In [5]:
model = GraphAttentionGCN(
    feature_names=feature_cols,
    edge_dim=2,
    num_layers=2,
    weight_decay=1e-4,
    patience=50,
    loss_name="huber",
    huber_delta=0.5,
    grad_clip_norm=5.0,
)

param_grid = {
    "hidden": [64, 96, 128],
    "num_heads": [2, 4, 6],
    "dropout": [0.10, 0.25, 0.45],
    "lr": [1e-3, 5e-4],
    "num_layers": [2, 3],
    "weight_decay": [1e-4, 1e-3, 1e-2],
    "loss_name": ["huber"],
    "huber_delta": [0.3, 0.5, 1.0],
    "grad_clip_norm": [1.0, 5.0, 7.0],
}

model.tune_hyperparameters(
    X_train,
    y_train,
    edge_index=edge_index_train,
    edge_attr=edge_attr_train,
    X_val=X_val,
    y_val=y_val,
    val_edge_index=edge_index_val_cross,
    val_edge_attr=edge_attr_val_cross,
    param_grid=param_grid,
    search_type="random",
    n_iter=7,
    optimize_metric="mape",
    epochs=600,
    refit_epochs=6000,
    random_state=42,
)

best_config = model.best_params_


Ejecutando 7 configuraciones (search_type=random, total_grid=2916).
Detectado grafo de validación cross-split; la evaluación usará el grafo combinado train+val y luego recortará solo los nodos target.
Entrenando config {'hidden': 64, 'num_layers': 3, 'num_heads': 2, 'dropout': 0.45, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'huber', 'huber_delta': 1.0, 'grad_clip_norm': 5.0}


/home/saneliges/Escritorio/PredictorPrecioMetroCuadradoAlquileres/ml_core/models/gat_gcn_model.py:309: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  y_tensor = torch.as_tensor(np.asarray(y).reshape(-1, 1), dtype=torch.float32, device=self.device)


Entrenando config {'hidden': 128, 'num_layers': 2, 'num_heads': 4, 'dropout': 0.45, 'lr': 0.001, 'weight_decay': 0.01, 'loss_name': 'huber', 'huber_delta': 0.3, 'grad_clip_norm': 1.0}
Entrenando config {'hidden': 96, 'num_layers': 3, 'num_heads': 2, 'dropout': 0.45, 'lr': 0.0005, 'weight_decay': 0.001, 'loss_name': 'huber', 'huber_delta': 0.3, 'grad_clip_norm': 1.0}
Entrenando config {'hidden': 96, 'num_layers': 2, 'num_heads': 6, 'dropout': 0.45, 'lr': 0.0005, 'weight_decay': 0.001, 'loss_name': 'huber', 'huber_delta': 1.0, 'grad_clip_norm': 5.0}
Entrenando config {'hidden': 64, 'num_layers': 3, 'num_heads': 2, 'dropout': 0.45, 'lr': 0.001, 'weight_decay': 0.001, 'loss_name': 'huber', 'huber_delta': 1.0, 'grad_clip_norm': 5.0}
Entrenando config {'hidden': 128, 'num_layers': 3, 'num_heads': 2, 'dropout': 0.45, 'lr': 0.0005, 'weight_decay': 0.001, 'loss_name': 'huber', 'huber_delta': 0.3, 'grad_clip_norm': 7.0}
Entrenando config {'hidden': 96, 'num_layers': 2, 'num_heads': 2, 'dropout':

## Entrenamiento

In [6]:
print("Mejor config:", best_config)

Mejor config: {'hidden': 128, 'num_layers': 3, 'num_heads': 2, 'dropout': 0.45, 'lr': 0.0005, 'weight_decay': 0.001, 'loss_name': 'huber', 'huber_delta': 0.3, 'grad_clip_norm': 7.0}


In [13]:
# Inicializar modelo con la mejor config encontrada


edge_dim = 2  # distancia km + distancia de features (localmente pesada)
model = GraphAttentionGCN(
    feature_names=feature_cols,
    edge_dim=edge_dim,
    hidden=best_config["hidden"],
    num_layers=best_config["num_layers"],
    num_heads=best_config["num_heads"],
    dropout=best_config["dropout"],
    lr=best_config["lr"],
    weight_decay=best_config["weight_decay"],
    patience=150,
    loss_name=best_config["loss_name"],
    huber_delta=best_config["huber_delta"],
    grad_clip_norm=best_config["grad_clip_norm"],
)



# Entrenamiento
_ = model.fit(
    X_train,
    y_train,
    edge_index=edge_index_train,
    edge_attr=edge_attr_train,
    epochs=6000,
)


## Evaluación global

In [14]:
X_val = gdf_val[feature_cols]
coords_val = gdf_val[coord_cols].to_numpy()
y_val = gdf_val[target_col]


y_pred_log = model.predict(
    X_val_cross,
    edge_index=edge_index_val_cross,
    edge_attr=edge_attr_val_cross,
)

y_pred = np.exp(y_pred_log)
y_true = np.exp(y_val_cross)

metrics = regression_metrics(y_true, y_pred)
metrics

{'rmse': 77661.42922800992,
 'mae': 42017.89505722952,
 'r2': 0.6131241505427829,
 'bias': 29405.194799600882,
 'median_abs_error': 21065.109374999898,
 'mape': 22.30529101812233}

## Diagnósticos específicos

Ideas para explorar:

- performance por zona
- sensibilidad a la definición del grafo
- ablations de features espaciales
- evolución de loss y early stopping

In [9]:
# history = pd.DataFrame(model.history_) if hasattr(model, "history_") else pd.DataFrame()
# history.tail()

## Embeddings / representaciones aprendidas

Si más adelante querés interpretación más rica, este bloque es un buen lugar para proyectar embeddings con UMAP/t-SNE y colorearlos por barrio, rango de precio o error.

In [10]:
# embeddings = None  # completar si expones el penultimo layer
# attention_summary = None  # completar si guardas pesos de atencion

## Export

In [11]:
# test_export = test_df[[target_col] + coord_cols].copy()
# test_export = test_export.rename(columns={target_col: "y_true"})
# test_export["y_pred"] = np.asarray(y_pred).reshape(-1)
# test_export["residual"] = test_export["y_true"] - test_export["y_pred"]
# test_export["split"] = "test"
# test_export.to_parquet(OUTPUT_DIR / "test_predictions.parquet", index=False)
# if embeddings is not None:
#     pd.DataFrame(embeddings).to_parquet(OUTPUT_DIR / "interpretability.parquet", index=False)
# (OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
# (OUTPUT_DIR / "run_config.json").write_text(json.dumps(gnn_config, indent=2, ensure_ascii=False))

## OOF Outliers

Refactor compartido con `12_rf_kriging`: la deteccion corre sobre folds y solo el armado del modelo/grafo queda especifico del notebook.


In [12]:
from ml_core.outlierAnalysis.oof import (
    detect_outliers_oof,
    load_active_processed_geodata,
)


def build_gnn_model():
    return GraphAttentionGCN(
        feature_names=feature_cols,
        edge_dim=2,
        hidden=int(best_config["hidden"]),
        num_layers=int(best_config["num_layers"]),
        num_heads=int(best_config["num_heads"]),
        dropout=0.55,
        lr=float(best_config["lr"]),
        weight_decay=float(best_config["weight_decay"]),
        patience=50,
        loss_name=str(best_config["loss_name"]),
        huber_delta=float(best_config["huber_delta"]),
        grad_clip_norm=float(best_config["grad_clip_norm"]),
    )


def build_local_weighted_graph(
    X_source,
    y_source,
    coords_source,
    *,
    X_target=None,
    coords_target=None,
    cross_split=False,
):

    coords_source = np.asarray(coords_source)
    coords_target = coords_source if coords_target is None else np.asarray(coords_target)
    X_target = X_source if X_target is None else X_target

    weights_source = compute_local_feature_weights(
        X_source,
        np.asarray(y_source).reshape(-1),
        coords_source,
        bandwidth_km=bandwidth_km,
        k_neighbors=k_neighbors,
    )

    if X_target is X_source and np.array_equal(coords_target, coords_source):
        weights_target = weights_source
    else:
        weights_target = project_local_feature_weights(
            weights_source,
            coords_source,
            coords_target,
            bandwidth_km=bandwidth_km,
            k_neighbors=k_neighbors,
        )

    graph_source = build_weighted_graph_frame(X_source, coords_source, weights_source)
    graph_target = build_weighted_graph_frame(X_target, coords_target, weights_target)

    builder = KNHS(
        lat_col="lat_deg",
        lon_col="lon_deg",
        feature_cols=feature_cols,
        weight_cols=weight_cols,
        distance="local_weighted",
        radius_km=radius_km,
        k=k_neighbors,
        add_reverse=True,
    )

    if cross_split and not (X_target is X_source and np.array_equal(coords_target, coords_source)):
        _, edge_index, edge_attr, _ = build_cross_split_knhs_graph(builder, graph_source, graph_target)
    else:
        edge_index, edge_attr = builder.build(graph_target)

    return {
        "edge_index": edge_index,
        "edge_attr": edge_attr,
    }


def gnn_fit_kwargs(context):
    return build_local_weighted_graph(
        context["X_train"],
        context["y_train"],
        context["coords_train"],
    )


def gnn_predict_kwargs(context):
    return build_local_weighted_graph(
        context["X_train"],
        context["y_train"],
        context["coords_train"],
        X_target=context["X_val"],
        coords_target=context["coords_val"],
    )


DATA_PATH = PROJECT_ROOT / "data" / "processed"

gdf_all = load_active_processed_geodata(
    data_path=DATA_PATH / "arg_venta_data_processed.csv",
    feature_cols=feature_cols,
    target_col=target_col,
    coord_cols=coord_cols,
    extra_cols=["idx", "url", "precio"],
)

X_all = gdf_all[feature_cols]
y_all = gdf_all[target_col]
coords_all = gdf_all[coord_cols].to_numpy()

results, residuals_oof = detect_outliers_oof(
    model_factory=build_gnn_model,
    X=X_all,
    y=y_all,
    gdf=gdf_all,
    coords=coords_all,
    output_dir=OUTPUT_DIR / "outliers_oof",
    methods=["ztest"],
    params_for_methods={
        "ztest": {"alpha": 0.05},
    },
    k_neighbors=15,
    n_splits=5,
    fit_kwargs_resolver=gnn_fit_kwargs,
    predict_kwargs_resolver=gnn_predict_kwargs,
)

results.head()




Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5


,idx,url,fold,method,precio,latitud,longitud,valido_hasta,pozo,estado_num,...,dist_est_educativo_scaled,dist_espacio_verde_scaled,dist_areas_programaticas_scaled,dist_avenida_rivadavia_scaled,n_robos_1000m_scaled,n_universidades_1000m_scaled,geometry,z_score,abs_z_score,tipo_valor_atipico
0,59423,https://www.argenprop.com/departamento-en-vent...,1,ztest,330000.0,-34.57038,-58.44061,NaN,0.0,1,...,0.314631,2.405715,-0.191892,0.874362,-0.587859,0.404804,POINT (-58.44061 -34.57038),12.988042,12.988042,ALTO
1,48472,https://www.argenprop.com/departamento-en-vent...,1,ztest,80000.0,-34.63431,-58.43269,NaN,0.0,1,...,0.325538,0.213971,-0.191892,-0.467298,-0.074591,-0.612433,POINT (-58.43269 -34.63431),9.175410,9.175410,ALTO
2,56936,https://www.argenprop.com/departamento-en-vent...,1,ztest,120000.0,-34.61077,-58.36746,NaN,0.0,3,...,1.979745,-1.398515,5.160811,-0.998644,0.186392,1.591580,POINT (-58.36746 -34.61077),-9.109588,9.109588,BAJO
3,46023,https://www.argenprop.com/departamento-en-vent...,1,ztest,79000.0,-34.63072,-58.44196,NaN,0.0,1,...,0.107033,-1.123449,-0.191892,-0.778800,0.482174,-0.442894,POINT (-58.44196 -34.63072),8.458958,8.458958,ALTO
4,44005,https://www.argenprop.com/departamento-en-vent...,1,ztest,395000.0,-34.59379,-58.38436,NaN,0.0,1,...,1.614002,-0.556829,-0.191892,-0.504093,0.960644,1.761120,POINT (-58.38436 -34.59379),7.736540,7.736540,ALTO
